In [ ]:
!pip install -r requirements.txt
!pip install networkx==3.5
# !pip install pyg-lib -f https://data.pyg.org/whl/torch-${TORCH}+${CUDA}.html
    # Where the last part is: torch-{python3 -c "import torch; print(torch.__version__)"}.html

# Evaluations

- Structural Statistics
	- Degree Distribution error (KL-divergence) over epsilon
	- Clustering coefficients errors (KL-divergence) over epsilon
	- Assortativity coefficients errors (KL-divergence) over epsilon
	- Centralities errors (KL-divergence) over epsilon
- Community Preservation
	- Modularity errors (KL-divergence) over epsilon
	- Adjusted Random Index over epsilon
	- Normalised Mutual Information over epsilon 
	- Distribution of Community Sizes error (KL-divergence)
	- After a greedy matching with communities from the ground truth, Jaccard Index for each community
- Social Recommendation Ranking Metrics
	- Normalised Discounted Cumulative Gain
	- (See edge prediction and community detection below)
- Accuracy/F1/etc. on Downstream Tasks (GNN / GCN / GraphSAGE Training):
	- Node Classification
	- Edge Prediction
	- Community Detection (Graph Clustering)

In [1]:
EPS = [0.5, 0.75, 1, 1.5, 2, 3, 4.5, 6.5, 9, 12, 16, 20]
DATAS = {
    "Facebook": {
        "feature_file": "feature_matrix_50_densest_binary-ish.txt",
        "target_file": "",
        "order": 0,
    },
    "LastFM": {
        "feature_file": "feature_matrix_50_highest_entropy.pkl",
        "target_file": "targets_reduced_5.txt",
        "order": 1,
    },
    "GitHub": {
        "feature_file": "feature_matrix_50_highest_entropy.pkl",
        "target_file": "targets.txt",
        "order": 2,
    },
    "Brightkite": {
        "feature_file": "feature_matrix_top_50.txt",
        "target_file": "",
        "order": 3,
    }
}
METHODS = {
    "ALJ": {
        "directory": "Ahmed16 - ALJ/result/",
        "num_trials": 10,
        "order": 2,
    },
    "CAGMDP": {
        "directory": "CAGMDP, TriCycLe/CAGMDP_result/", # 2020
        "num_trials": 10,
        "order": 5,
    },
    "TriCycLe": {
        "directory": "CAGMDP, TriCycLe/TriCycLe_result/", # 2016
        "num_trials": 10,
        "order": 3,
    },
    "DER": {
        "directory": "Chen14 - DER/result/",
        "num_trials": 10,
        "order": 0,
    },
    "TmF": {
        "directory": "Nguyen15 - TmF/result/",
        "num_trials": 10,
        "order": 1,
    },
    "LDPGen": {
        "directory": "Qin17 - LDPGen/result/",
        "num_trials": 10,
        "order": 4,
    },
    "DPGGAN": {
        "directory": "Yang21 - DPGGAN, DPGVAE/DPGGAN_result/",
        "num_trials": 10,
        "order": 7,
    },
    "DPGVAE": {
        "directory": "Yang21 - DPGGAN, DPGVAE/DPGVAE_result/",
        "num_trials": 10,
        "order": 6,
    },
    "PrivGraph": {
        "directory": "Yuan23 - PrivGraph/result/",
        "num_trials": 10,
        "order": 8,
    },
    "PrivDPR": {
        "directory": "Zhang25 - PrivDPR/result/",
        "num_trials": 10,
        "order": 9,
    },
}

In [2]:
worker = 0
tot_worker = 1

import os, sys, math, random, gc, pickle
#worker = int(sys.argv[1])
#tot_worker = int(sys.argv[2])
gpu_id = ""
if len(sys.argv) >= 4:
    gpu_id = sys.argv[3]
os.environ["CUDA_VISIBLE_DEVICES"] = gpu_id
if gpu_id:
    os.environ["NX_CUGRAPH_AUTOCONFIG"] = "True"
else:
    os.environ["NX_CUGRAPH_AUTOCONFIG"] = "False" 
    os.environ["NETWORKX_AUTOMATIC_BACKENDS"] = "parallel"
os.environ["OMP_NUM_THREADS"] = str(os.cpu_count() // tot_worker)
os.environ["MKL_NUM_THREADS"] = str(os.cpu_count() // tot_worker)
os.environ["OPENBLAS_NUM_THREADS"] = str(os.cpu_count() // tot_worker)
os.environ["NUMEXPR_NUM_THREADS"] = str(os.cpu_count() // tot_worker)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(os.cpu_count() // tot_worker)
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import networkit as nk
from tqdm import tqdm
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score, ndcg_score
from scipy.stats import wasserstein_distance
import netmax.influence_maximization as im
from collections import Counter
from datetime import datetime
from threading import Thread
from glob import glob
from multiprocessing import Pool
from pprint import pprint
import pandas as pd
pd.set_option('display.width', 200)
from collections.abc import Mapping
from gcn_tasks import unprivate_link_predict, unprivate_node_classify, private_link_predict, private_node_classify

nk.engineering.setNumberOfThreads(os.cpu_count() // tot_worker)
nx.config.backends.parallel.active = True
nx.config.backends.parallel.n_jobs = os.cpu_count() // tot_worker

def merge_dicts_to_lists(dicts):
    def merge(*objs):
        if all(isinstance(obj, Mapping) for obj in objs):
            keys = set().union(*objs)
            return {
                key: merge(*(obj.get(key) for obj in objs))
                for key in keys
            }
        else: # Base case: lowest-level value → collect into list
            return list(objs)
    return merge(*dicts)

def exact_harmonic_diameter(G):
    n = G.numberOfNodes()
    sum_reciprocal_distances = 0
    count_pairs = 0
    def bfs(u):
        nonlocal sum_reciprocal_distances, count_pairs
        x = nk.distance.BFS(G, u, storePaths=False)
        x.run()
        dists = x.getDistances()
        for v in range(n):
            if u != v:
                d = dists[v]
                if d < float('inf'):
                    sum_reciprocal_distances += 1 / d
                    count_pairs += 1
    for i in tqdm(range(0,n,os.cpu_count()), desc="Harmonic Diameter. Running BFS"):
        threads = []
        for j in range(os.cpu_count()):
            if i+j >= n:
                break
            threads.append(Thread(target=bfs, args=(i+j,)))
            threads[j].start()
        for j in range(len(threads)):
            threads[j].join()
    if count_pairs == 0:
        return float('inf')  # completely disconnected
    harmonic_diameter = n * (n - 1) / sum_reciprocal_distances
    return harmonic_diameter
    

def single_metrics(G):
    Gk = nk.nxadapter.nx2nk(G)
    degseq = [d for _, d in G.degree()]
    degcnt = Counter(degseq)
    if len(G.nodes()) > 10000: # approximations
        x = nk.centrality.ApproxBetweenness(Gk, epsilon=0.01, delta=0.05) # https://dl.acm.org/doi/abs/10.1145/2556195.2556224
        x.run()
        betweenness = x.scores()
        x = nk.centrality.ApproxCloseness(Gk, nSamples=1000, epsilon=0.01, normalized=True) #https://arxiv.org/pdf/1409.0035
        x.run()
        closeness = x.scores()
        # leskovec, 2006, Sampling from Large Graphs, KDD'06
        # https://www.sciencedirect.com/science/article/abs/pii/S0378437100003113
    else:
        betweenness = [y for x,y in sorted((v,c) for v,c in nx.betweenness_centrality(G).items())]
        closeness = [y for x,y in sorted((v,c) for v,c in nx.closeness_centrality(G).items())]
    h_diameter = exact_harmonic_diameter(Gk)
    communities = nx.community.louvain_communities(G)
    community_vector = [None for _ in range(max(G.nodes())+1)]
    for com_id, com_set in enumerate(communities):
        for v in com_set:
            community_vector[v] = com_id
    x = nk.centrality.PageRank(Gk, damp=0.85, tol=1e-6)
    x.run()
    pageranks = x.scores()
    del Gk; gc.collect()
    
    return {
        'local': { # node-wise
            'Degree Distribution': [degcnt.get(i, 0) for i in range(max(degcnt)+1)],
            'Clustering Coefficient Distribution': [y for x,y in sorted((v,c) for v,c in nx.clustering(G).items())],
            'Betweenness Centrality Distribution': betweenness,
            'Closeness Centrality Distribution': closeness,
        },
        'global': {
            'Density': nx.density(G),
            'Harmonic Diameter': h_diameter,
            'Assortativity': nx.degree_assortativity_coefficient(G),
            'Modularity': nx.community.modularity(G, communities),
            'Transitivity': nx.transitivity(G),
        },
        'meta': {
            'community_vector': community_vector, # For ARI, NMI
            'pagerank_vector': pageranks # For NDCG
        },
    }


def community_vector2partition(community_vector):
    partition = [set() for _ in range(max(community_vector)+1)]
    for v, com in enumerate(community_vector):
        partition[com].add(v)
    return partition


def comparative_metrics(G1_dict, G2_dict):
    G1_community_vector = G1_dict['meta']['community_vector']
    G2_community_vector = G2_dict['meta']['community_vector']
    next_id = max(G1_community_vector)+1
    while len(G1_community_vector) < len(G2_community_vector):
        G1_community_vector.append(next_id)
        next_id += 1
    while len(G1_community_vector) > len(G2_community_vector):
        G2_community_vector.append(next_id)
        next_id += 1
    assert len(G1_community_vector) == len(G2_community_vector)
    return {
        'ARI': adjusted_rand_score(G1_community_vector, G2_community_vector),
        'NMI': normalized_mutual_info_score(G1_community_vector, G2_community_vector), 
        'NDCG over PageRank Scores': ndcg_score(G1_dict['meta']['pagerank_vector'], G2_dict['meta']['pagerank_vector']),
    }


def influence_maximisation(G): # DO RELATIVE ERROR FOR THIS. IT's LIKE A GLOBAL STATISTIC.
    # TIM+ method (NOT degree-discount)
    # Independent Cascade.
    # see the GitHub NetMax repo for references.
    D = G.to_directed()
    im_instance = im.InfluenceMaximization(input_graph=D, agents={"0":int(D.number_of_nodes() * 0.01)}, alg='tim_p', 
                                            diff_model='ic', inf_prob='uniform', r=1000,
                                            insert_opinion=False, endorsement_policy='random', verbose=False)
    seed, spread, execution_time = im_instance.run()
    return spread

## Unprivate Data Statistics

This should only need to be run once for every raw dataset.

In [3]:
def unprivate_evaluation(dataset):
    with open(f"../Data/{dataset}/nx_adj.pkl", 'rb') as f:
        G = pickle.load(f)
    results = dict()
    results = single_metrics(G)
    results["utility"] = dict()
    results["utility"]["Influence Maximisation"] = influence_maximisation(G)
    scores, edges_val, neg_val, edges_test, neg_test = unprivate_link_predict.run(f"../Data/{dataset}/nx_adj.pkl", f"../Data/{dataset}/{DATAS[dataset]['feature_file']}")#, use_node2vec=True)
    results["utility"]["Link Prediction"] = scores
    results["meta"]["Link Prediction"] = dict()
    results["meta"]["Link Prediction"]["val_pos"] = edges_val
    results["meta"]["Link Prediction"]["val_neg"] = neg_val
    results["meta"]["Link Prediction"]["test_pos"] = edges_test
    results["meta"]["Link Prediction"]["test_neg"] = neg_test
    if dataset in ["LastFM", "GitHub"]:
        scores, val_mask, test_mask = unprivate_node_classify.run(f"../Data/{dataset}/nx_adj.pkl", f"../Data/{dataset}/{DATAS[dataset]['feature_file']}", f"../Data/{dataset}/{DATAS[dataset]['target_file']}", use_node2vec=True)
        results["utility"]["Node Classification"] = scores
        results["meta"]["Node Classification"] = dict()
        results["meta"]["Node Classification"]["val_mask"] = val_mask
        results["meta"]["Node Classification"]["test_mask"] = test_mask
    with open(f"./Unprivate/{dataset}.pkl", 'wb') as f:
        pickle.dump(results, f)

for dataset in ["LastFM", "GitHub"]:#, "GitHub", "Brightkite"]: #DATAS.keys(): #0.5059, F1: 0.2522 node2vec reduced
    unprivate_evaluation(dataset)

GCN, Epoch:   5%|███▋                                                                | 27/500 [01:16<22:21,  2.84s/it]

Stopping early.


Test Accuracy: 0.5001, ROC AUC: 0.7930, AP: 0.7910


GCN, Epoch:  19%|█████████████▏                                                      | 97/500 [01:42<07:07,  1.06s/it]

Stopping early.


...

Stopping early.


Test Accuracy: 0.5000, ROC AUC: 0.9134, AP: 0.9235


GCN, Epoch:  21%|██████████████▎                                                    | 107/500 [03:47<13:55,  2.13s/it]

Stopping early.


Accuracy: 0.6552, F1: 0.6207


In [76]:
from pprint import pprint

for dataset in DATAS.keys():
    print(dataset)
    with open(f'./Unprivate/{dataset}.pkl', 'rb') as f:
        result = pickle.load(f)
    with open(f'../Data/{dataset}/nx_adj.pkl', 'rb') as f:
        g = pickle.load(f)
    print(g.number_of_nodes(), g.number_of_edges())
    pprint(result["global"])
    pprint(result["utility"])
    print()

Facebook
4039 88234
{'Assortativity': 0.06357722918564943,
 'Density': 0.010819963503439287,
 'Harmonic Diameter': 3.261811080029313,
 'Modularity': 0.8348200533979873,
 'Transitivity': 0.5191742775433075}
{'Influence Maximisation': {'0': 2371.684},
 'Link Prediction': {'Accuracy': 0.8100702708829196,
                     'Average Precision': 0.9611368645504479,
                     'ROC AUC': 0.9665597614084204}}

LastFM
7624 27806
{'Assortativity': 0.017073172560631417,
 'Density': 0.0009568849118596328,
 'Harmonic Diameter': 4.874223397958484,
 'Modularity': 0.8142873207073501,
 'Transitivity': 0.178622548153384}
{'Influence Maximisation': {'0': 735.485},
 'Link Prediction': {'Accuracy': 0.5000899118863513,
                     'Average Precision': 0.7909576423721503,
                     'ROC AUC': 0.7929995552748884},
 'Node Classification': {'Accuracy': 0.42857142857142855,
                         'F1 Score': 0.24255341652041396}}

GitHub
37700 289003
{'Assortativity': -0.075217

## Private Measurements

Evaluates metrics and stores as lists of trials.

In [3]:
def private_evaluation(dataset, method, epsilon, with_influence_maximisation=True):
    results_list = list()
    trials = glob(f"../{METHODS[method]['directory']}/{dataset}/nx_adj_eps{epsilon}_i*.pkl")
    for i, path in enumerate(trials):
        print(f"Evaluating {dataset}, {method}, Epsilon {epsilon}, Trial {i}.")
        with open(path, 'rb') as f:
            G = pickle.load(f)
        results = single_metrics(G)
        G.__networkx_cache__.clear()
        results["utility"] = dict()
        if with_influence_maximisation:
            results["utility"]["Influence Maximisation"] = influence_maximisation(G)
        else:
            results["utility"]["Influence Maximisation"] = list()
        with open(f"./Unprivate/{dataset}.pkl", 'rb') as f:
            unprivate = pickle.load(f)
            val_pos = unprivate["meta"]["Link Prediction"]["val_pos"]
            val_neg = unprivate["meta"]["Link Prediction"]["val_neg"]
            test_pos = unprivate["meta"]["Link Prediction"]["test_pos"]
            test_neg = unprivate["meta"]["Link Prediction"]["test_neg"]
        if method in ["CAGMDP", "TriCycLe"]:
            feature_file = f"../CAGMDP, TriCycLe/{method}_result/{dataset}/np_att_eps{epsilon}_i{i}.txt"
        else:
            feature_file = f"../Data/{dataset}/{DATAS[dataset]['feature_file']}"
        scores = private_link_predict.run(f"../Data/{dataset}/nx_adj.pkl", feature_file, val_pos, val_neg, test_pos, test_neg)
        results["utility"]["Link Prediction"] = scores
        if dataset in ["LastFM", "GitHub"]:
            with open(f"./Unprivate/{dataset}.pkl", 'rb') as f:
                unprivate = pickle.load(f)
                val_mask = unprivate["meta"]["Node Classification"]["val_mask"]
                test_mask = unprivate["meta"]["Node Classification"]["test_mask"]
            scores = private_node_classify.run(f"../Data/{dataset}/nx_adj.pkl", feature_file, f"../Data/{dataset}/{DATAS[dataset]['target_file']}", val_mask, test_mask, use_node2vec=True)
            results["utility"]["Node Classification"] = scores
        results_list.append(results)
    measurements = merge_dicts_to_lists(results_list)
    with open(f"./Measurements/{dataset}/{method}_eps{epsilon}.pkl", 'wb') as f:
        pickle.dump(measurements, f)
    print('\n\n')

In [ ]:
for dataset in ["Facebook"]:#DATAS.keys():
    for method in METHODS.keys():
        for epsilon in EPS:
            if os.path.exists(f"./Measurements/{dataset}/{method}_eps{epsilon}.pkl"):
                print(f"Skipping {dataset}, {method}, Epsilon {epsilon}. File exists.")
                continue
            else:
                private_evaluation(dataset, method, epsilon)

### Parallelised Version

Omits Influence Maximization because this cannot be parallelized easily. This should instead be done later.

In [ ]:
# Assuming backgrounded from the CLI. (GPU set in the command line)
# Must be launched like: CUDA_VISIBLE_DEVICES=0 python3 script.py 0 0 0
# first number is worker_id, second is total_workers, third is gpu to use.
print(f"Worker {worker} out of {tot_worker}. Using GPU {gpu_id if gpu_id else 'None'}.")
dataset = "LastFM"
methods = [method for i, method in enumerate(METHODS.keys()) if i % tot_worker == worker]
print(methods)
for method in methods:
    for epsilon in EPS:
        if os.path.exists(f"./Measurements/{dataset}/{method}_eps{epsilon}.pkl"):
            print(f"Skipping {dataset}, {method}, Epsilon {epsilon}. File exists.")
            continue
        else:
            private_evaluation(dataset, method, epsilon, with_influence_maximisation=False)
        gc.collect()


################################ NEW STUFF: DO INFLUENCE MAXIMIZATION LAST.
from multiprocessing import Pool
def influence_maximisation_for_parallel(dataset, method, epsilon, trial_id):
    with open(f"../{METHODS[method]['directory']}/{dataset}/nx_adj_eps{epsilon}_i{trial_id}.pkl", 'rb') as f:
        G = pickle.load(f)
    D = G.to_directed()
    im_instance = im.InfluenceMaximization(
        input_graph=D, agents={"0":int(D.number_of_nodes() * 0.01)}, alg='tim_p', diff_model='ic', 
        inf_prob='uniform', r=1000, insert_opinion=False, endorsement_policy='random', verbose=False
    )
    seed, spread, execution_time = im_instance.run()
    return spread

jobs = []
for method in methods:
    for epsilon in EPS:
        with open(f"./Measurements/{dataset}/{method}_eps{epsilon}.pkl", 'rb') as f:
            result = pickle.load(f)
        if result["utility"]["Influence Maximisation"]:
            continue
        for i in range(METHODS[method]["num_trials"]):
            jobs.append((dataset, method, epsilon, i))

with Pool(processes=os.cpu_count()//tot_worker) as pool:
    results = pool.starmap(influence_maximisation_for_parallel, jobs)

for (dataset, method, epsilon, i), res in zip(jobs, results):
    curr_path = f"./Measurements/{dataset}/{method}_eps{epsilon}.pkl"
    with open(curr_path, 'rb') as f:
        result = pickle.load(f)
    result["utility"]["Influence Maximisation"].append(res)
    with open(curr_path, 'wb') as f:
        pickle.dump(result, f)

## Results

In [3]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="The number of unique classes is greater than 50% of the number of samples.*",
    category=UserWarning,
    module="sklearn.metrics.cluster._supervised"
)


def comparative_metrics_with_trials(G1_dict, G2_dict, n_trials): # G2 has the lists.
    result = {
        'ARI': 0,
        'NMI': 0,
        'NDCG over PageRank Scores': 0
    }    
    for i in range(n_trials):
        G1_community_vector = G1_dict['meta']['community_vector']
        G2_community_vector = G2_dict['meta']['community_vector'][i]
        # Correct G1 communities
        next_id = max(v for v in G1_community_vector if v is not None)+1
        while len(G1_community_vector) < len(G2_community_vector):
            G1_community_vector.append(next_id)
            next_id += 1
        for j in range(len(G1_community_vector)):
            if G1_community_vector[j] is None:
                G1_community_vector[j] = next_id
                next_id += 1
        # Correct G2 communities
        next_id = max(v for v in G2_community_vector if v is not None)+1
        while len(G1_community_vector) > len(G2_community_vector):
            G2_community_vector.append(next_id)
            next_id += 1
        for j in range(len(G2_community_vector)):
            if G2_community_vector[j] is None:
                G2_community_vector[j] = next_id
                next_id += 1
        # Correct Pageranks
        G1_pageranks = G1_dict['meta']['pagerank_vector']
        G2_pageranks = G2_dict['meta']['pagerank_vector'][i]
        while len(G1_pageranks) < len(G2_pageranks):
            G1_pageranks.append(0)
        while len(G1_pageranks) > len(G2_pageranks):
            G2_pageranks.append(0)
        assert len(G1_community_vector) == len(G2_community_vector)
        assert len(G1_dict['meta']['pagerank_vector']) == len(G2_dict['meta']['pagerank_vector'][i])
        result['ARI'] += adjusted_rand_score(G1_community_vector, G2_community_vector) / n_trials
        result['NMI'] += normalized_mutual_info_score(G1_community_vector, G2_community_vector) / n_trials
        result['NDCG over PageRank Scores'] += ndcg_score([G1_pageranks], [G2_pageranks]) / n_trials
    return result


def compute_error_dicts(dataset, methods):
    with open(f"./Unprivate/{dataset}.pkl", 'rb') as f:
        O = pickle.load(f) # dict
    methods.sort(key=lambda x: METHODS[x]["order"])
    print(methods)
    for method in methods:
        #if dataset == "Brightkite" and method in ["DPGGAN", "DPGVAE"]: continue # REMOVE ONCE FINISHED.
        results = {
            "local": dict(),
            "global": dict(),
            "comparative": dict(),
            "utility": dict()
        }
        for eps in EPS:
            with open(f"./Measurements/{dataset}/{method}_eps{eps}.pkl", 'rb') as f:
                G = pickle.load(f)
            for metric in G["local"].keys(): # local: wasserstein distances
                if metric not in results["local"].keys():
                    results["local"][metric] = list()
                avg = 0
                for i in range(METHODS[method]["num_trials"]):
                    avg += wasserstein_distance(
                                [v for v in O["local"][metric] if (not np.isnan(v)) and (not np.isinf(v))], 
                                [v for v in G["local"][metric][i] if (not np.isnan(v)) and (not np.isinf(v))]
                           ) / METHODS[method]["num_trials"]
                results["local"][metric].append(avg)
            for metric in G["global"].keys(): # global: absolute error, except harmonic diameter.
                if metric not in results["global"].keys():
                    results["global"][metric] = list()
                avg = 0
                for i in range(METHODS[method]["num_trials"]):
                    if metric == "Harmonic Diameter": # relative
                        avg += (abs(O["global"][metric] - G["global"][metric][i]) / abs(O["global"][metric])) / METHODS[method]["num_trials"]
                    else: # absolute
                        avg += (abs(O["global"][metric] - G["global"][metric][i])) / METHODS[method]["num_trials"]
                results["global"][metric].append(avg)
            comps = comparative_metrics_with_trials(O, G, METHODS[method]["num_trials"]) # comparative
            for metric in comps.keys():
                if metric not in results["comparative"].keys():
                    results["comparative"][metric] = list()
                results["comparative"][metric].append(comps[metric])
            # Influence Maximisation
            #"""
            metric = "Influence Maximisation"
            if metric not in results["utility"].keys():
                results["utility"][metric] = list()
            # INFLUENCE MAXIMISATION-SPECIFIC EXCLUSIONS
            if (dataset == "GitHub" and method == "LDPGen" and eps in [0.5, 0.75]) or \
               (dataset == "Brightkite" and method == "LDPGen" and eps in [0.5, 0.75]) or \
               (dataset in ["GitHub", "Brightkite"] and method == "PrivDPR"):
                results["utility"][metric].append(None)
            else:
                avg = 0
                for i in range(METHODS[method]["num_trials"]): # absolute error of %
                    o_percentage = O["utility"][metric]['0'] / len(O['meta']['pagerank_vector'])
                    g_percentage = G["utility"][metric][i] / len(G['meta']['pagerank_vector'][i])
                    avg += abs(o_percentage - g_percentage) / METHODS[method]["num_trials"]
                results["utility"][metric].append(avg)
            #"""
            # Link Prediction / Node Classification: absolute errors
            tasks = ["Link Prediction"]
            if dataset in ["LastFM", "GitHub"]:
                tasks.append("Node Classification")
            for task in tasks:
                if task not in results["utility"].keys():
                    results["utility"][task] = dict()
                for metric in G["utility"][task]:
                    if metric not in results["utility"][task].keys():
                        results["utility"][task][metric] = list()
                    avg = 0
                    for i in range(METHODS[method]["num_trials"]):
                        avg += abs(O["utility"][task][metric] - G["utility"][task][metric][i]) / METHODS[method]["num_trials"]
                    results["utility"][task][metric].append(avg)
        with open(f"./Results/Errors/dicts/{dataset}_{method}.pkl", 'wb') as f:
            pickle.dump(results, f)
        #print()

In [4]:
for dataset in ["Facebook", "LastFM", "GitHub", "Brightkite"]:
    compute_error_dicts(dataset, [method for method in METHODS.keys()]) # 

['DER', 'TmF', 'ALJ', 'TriCycLe', 'LDPGen', 'CAGMDP', 'DPGVAE', 'DPGGAN', 'PrivGraph', 'PrivDPR']
['DER', 'TmF', 'ALJ', 'TriCycLe', 'LDPGen', 'CAGMDP', 'DPGVAE', 'DPGGAN', 'PrivGraph', 'PrivDPR']
['DER', 'TmF', 'ALJ', 'TriCycLe', 'LDPGen', 'CAGMDP', 'DPGVAE', 'DPGGAN', 'PrivGraph', 'PrivDPR']
['DER', 'TmF', 'ALJ', 'TriCycLe', 'LDPGen', 'CAGMDP', 'DPGVAE', 'DPGGAN', 'PrivGraph', 'PrivDPR']


In [5]:
def tabulate_error_dicts(dataset, methods=[method for method in METHODS.keys()]):
    big_table = dict()
    methods.sort(key=lambda x: METHODS[x]["order"])
    for method in methods:
        #if dataset == "Brightkite" and method in ["DPGGAN", "DPGVAE"]: continue # REMOVE ONCE FINISHED.
        with open(f"./Results/Errors/dicts/{dataset}_{method}.pkl", 'rb') as f:
            result = pickle.load(f)
        for metric in result["local"].keys():
            if metric not in big_table.keys():
                big_table[metric] = {'epsilon': [eps for eps in EPS]}
            big_table[metric][method] = result["local"][metric]
        for metric in result["global"].keys():
            if metric not in big_table.keys():
                big_table[metric] = {'epsilon': [eps for eps in EPS]}
            big_table[metric][method] = result["global"][metric]
        for metric in result["comparative"].keys():
            if metric not in big_table.keys():
                big_table[metric] = {'epsilon': [eps for eps in EPS]}
            big_table[metric][method] = result["comparative"][metric]
        for task in result["utility"].keys():
            if task == "Influence Maximisation":
                if task not in big_table.keys():
                    big_table[task] = {'epsilon': [eps for eps in EPS]}
                big_table[task][method] = result["utility"][task]
            else: # Link Prediction / Node Classification
                for metric in result["utility"][task]:
                    header = task + " " + metric
                    if header not in big_table.keys():
                        big_table[header] = {'epsilon': [eps for eps in EPS]}
                    big_table[header][method] = result["utility"][task][metric]
    with open(f"./Results/Errors/dicts/{dataset}.pkl", 'wb') as f:
        pickle.dump(big_table, f)
    for k in big_table.keys():
        df = pd.DataFrame(big_table[k])
        print(k)
        print(df)
        df.to_csv(f"./Results/Errors/csvs/{dataset}_{k.replace(' ', '_')}.csv", index=False)
        print()

In [6]:
for dataset in ["Facebook", "LastFM", "GitHub", "Brightkite"]:
    tabulate_error_dicts(dataset)

Clustering Coefficient Distribution
    epsilon       DER       TmF       ALJ  TriCycLe    LDPGen    CAGMDP    DPGVAE    DPGGAN  PrivGraph   PrivDPR
0      0.50  0.382130  0.594453  0.575122  0.362886  0.275091  0.412759  0.481303  0.486239   0.564139  0.603853
1      0.75  0.383774  0.594326  0.575113  0.354682  0.436360  0.423497  0.481303  0.486239   0.542829  0.603753
2      1.00  0.387976  0.594130  0.575121  0.367432  0.491861  0.420001  0.481694  0.493122   0.494068  0.603832
3      1.50  0.394355  0.593503  0.575120  0.380165  0.510554  0.392060  0.490854  0.498158   0.470592  0.603734
4      2.00  0.401015  0.592312  0.575136  0.371494  0.507670  0.400655  0.490274  0.496698   0.465136  0.603739
5      3.00  0.411707  0.586107  0.575130  0.380957  0.500094  0.344876  0.502640  0.497841   0.466223  0.603761
6      4.50  0.424347  0.538616  0.575117  0.385995  0.502089  0.366414  0.508559  0.493579   0.457979  0.603841
7      6.50  0.436165  0.363823  0.575103  0.389363  0.50882

In [10]:
def tabulate_unprivate():
    table = dict({'Metric': []})
    for dataset in DATAS.keys():
        table[dataset] = []
        with open(f"./Unprivate/{dataset}.pkl", 'rb') as f:
            d = pickle.load(f)
        print(dataset)
        pprint(d["global"])
        pprint(d["utility"])
        print()
        for metric in sorted(d["global"].keys()):
            if dataset == "Facebook":
                table['Metric'].append(metric)
            table[dataset].append(d["global"][metric])
        if dataset == "Facebook":
            table['Metric'].append("Influence Maximisation Spread")
        table[dataset].append(d["utility"]["Influence Maximisation"]['0'] / len(d["meta"]["pagerank_vector"]))
        for inner in sorted(d["utility"]["Link Prediction"].keys()):
            metric = "Link Prediction " + inner
            if dataset == "Facebook":
                table["Metric"].append(metric)
            table[dataset].append(d["utility"]["Link Prediction"][inner])
        if dataset in ["LastFM", "GitHub"]:
            for inner in sorted(d["utility"]["Node Classification"].keys()):
                metric = "Node Classification " + inner
                if dataset == "LastFM":
                    table['Metric'].append(metric)
                table[dataset].append(d["utility"]["Node Classification"][inner])
        else:
            table[dataset].append(None)
            table[dataset].append(None)
            
    df = pd.DataFrame(table)
    print(df)
    df.to_csv("./Unprivate/unprivate.csv", index=False)

tabulate_unprivate()

Facebook
{'Assortativity': 0.06357722918564943,
 'Density': 0.010819963503439287,
 'Harmonic Diameter': 3.261811080029313,
 'Modularity': 0.8348200533979873,
 'Transitivity': 0.5191742775433075}
{'Influence Maximisation': {'0': 2371.684},
 'Link Prediction': {'Accuracy': 0.8100702708829196,
                     'Average Precision': 0.9611368645504479,
                     'ROC AUC': 0.9665597614084204}}

LastFM
{'Assortativity': 0.017073172560631417,
 'Density': 0.0009568849118596328,
 'Harmonic Diameter': 4.874223397958484,
 'Modularity': 0.8142873207073501,
 'Transitivity': 0.178622548153384}
{'Influence Maximisation': {'0': 735.485},
 'Link Prediction': {'Accuracy': 0.5000899118863513,
                     'Average Precision': 0.7909576423721503,
                     'ROC AUC': 0.7929995552748884},
 'Node Classification': {'Accuracy': 0.42857142857142855,
                         'F1 Score': 0.24255341652041396}}

GitHub
{'Assortativity': -0.07521713413904482,
 'Density': 0.00040668